# ProphetGP 사용 예시

이 노트북은 ProphetGP의 핵심 기능(학습, 후보 추천, 데이터셋 추가)을 빠르게 실행해보는 예시입니다.

In [1]:
# 필요 시 주석 해제 후 설치
# !pip install -e .[dev]

In [1]:
import numpy as np
from pathlib import Path

from prophet_gp.config import load_config
from prophet_gp.pipeline.trainer import ProphetGPPipeline
from prophet_gp.data.dataset import ReactionDatasetService

# 노트북 실행 위치가 notebooks/여도 안전하게 프로젝트 루트를 찾는다.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "configs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

CONFIG_PATH = PROJECT_ROOT / "configs" / "test_20260530.yaml"
DATA_PATH = PROJECT_ROOT / "data" / "sample" / "sample_data_extended.csv"
NEW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "new_batch.csv"
MERGED_OUT_PATH = PROJECT_ROOT / "data" / "raw" / "reactions_merged.csv"

config = load_config(CONFIG_PATH)
pipeline = ProphetGPPipeline(config)
dataset_service = ReactionDatasetService(config.data)

print("Project root:", PROJECT_ROOT)
print("Config loaded:", config.model_dump())

Project root: c:\Projects\ProphetGP
Config loaded: {'data': {'reactant_column': 'Mol.1', 'target_column': ['Emission Peak', 'FWHM'], 'reactant_delimiter': '|', 'ignore_columns': ['PLQY', 'Pristine Emission', 'Pristine FWHM', 'Pristine PLQY'], 'reactant_allowed_values': [], 'condition_ranges': {'Temperature': {'min': 0.0, 'max': 600.0, 'allowed_values': None}}, 'explicit_condition_types': {'Temperature': 'continuous'}}, 'featurization': {'featuriser': 'morgan_fp', 'combine_strategy': 'concat'}, 'optimization': {'objective': 'target', 'suggestion_strategy': 'best_output', 'standardize_gp_inputs': False, 'standardize_gp_targets': False, 'target_value': None, 'target_objectives': {'Emission Peak': {'objective': 'target', 'target_value': 490.0, 'weight': 1.0}, 'FWHM': {'objective': 'minimize', 'target_value': None, 'weight': 0.8}}, 'n_restarts': 10, 'raw_samples': 128, 'target_search_size': 5000, 'n_candidates': 3}}


In [2]:
# 1) 사용 가능한 featuriser 확인
available_featurisers = pipeline.featurizers.available()
print("Available featurisers count:", len(available_featurisers))
print(available_featurisers[:20])  # 앞쪽 일부만 출력

Available featurisers count: 10
['bag_of_characters', 'drfp', 'ecfp_fingerprints', 'fragments', 'molecular_graphs', 'morgan_fp', 'mqn_features', 'one_hot', 'rdkit_descriptors', 'rxnfp']


In [9]:
# 2) 학습
artifacts = pipeline.train_from_csv(DATA_PATH)
print("Train rows:", artifacts.x_train.shape[0])
print("Feature dims:", artifacts.x_train.shape[1])
print("molecular representation check:", np.unique(artifacts.x_train[:, :-1], axis=0).shape)
print("rank of training data:", np.linalg.matrix_rank(artifacts.x_train))

c:\Users\sung1234\AppData\Local\Programs\Python\Python39\lib\site-packages\botorch\models\utils\assorted.py:174: InputDataWarning: Input data is not contained to the unit cube. Please consider min-max scaling the input data.
  warnings.warn(msg, InputDataWarning)
c:\Users\sung1234\AppData\Local\Programs\Python\Python39\lib\site-packages\botorch\models\utils\assorted.py:202: InputDataWarning: Input data is not standardized (mean = tensor([514.3433], dtype=torch.float64), std = tensor([64.5160], dtype=torch.float64)). Please consider scaling the input to zero mean and unit variance.
  warnings.warn(msg, InputDataWarning)
c:\Users\sung1234\AppData\Local\Programs\Python\Python39\lib\site-packages\botorch\models\utils\assorted.py:202: InputDataWarning: Input data is not standardized (mean = tensor([97.7463], dtype=torch.float64), std = tensor([31.2048], dtype=torch.float64)). Please consider scaling the input to zero mean and unit variance.
  warnings.warn(msg, InputDataWarning)


Train rows: 67
Feature dims: 1025
molecular representation check: (38, 1024)
rank of training data: 32


In [5]:
# 3) 다음 실험 조건 후보 추천 (raw + 해석 결과)
# strategy: "best_output" | "best_information"
n_candidates = 3
strategy = "best_output"
suggestions = pipeline.suggest_next_experiments(
    artifacts,
    n_candidates=n_candidates,
    strategy=strategy,
)

print("Strategy:", strategy)
print("Raw candidates shape:", suggestions.raw_candidates.shape)
print("Decoded candidates:")
for idx, row in enumerate(suggestions.decoded_candidates, 1):
    print(f"- candidate_{idx}")
    print("  predicted mean:", row["predicted_target_mean"])
    print("  predicted std:", row["predicted_target_std"])
    print("  target gap:", row["target_gap"])
    print("  nearest_known_reactants_input:", row["nearest_known_reactants_input"])
    print("  Temperature:", row["Temperature"])
    print("  mapped input:", row)

# suggestions.raw_candidates

Strategy: best_output
Raw candidates shape: (3, 1025)
Decoded candidates:
- candidate_1
  predicted mean: {'Emission Peak': 514.3841710320966, 'FWHM': 97.73372279315683}
  predicted std: {'Emission Peak': 3.228677382121188, 'FWHM': 3.135026945362148}
  target gap: {'Emission Peak': 24.38417103209656, 'FWHM': None}
  nearest_known_reactants_input: 5-amino-1,10-phenanthroline|salicylic acid
  Temperature: 91.59333801269531
  mapped input: {'predicted_target_mean': {'Emission Peak': 514.3841710320966, 'FWHM': 97.73372279315683}, 'predicted_target_std': {'Emission Peak': 3.228677382121188, 'FWHM': 3.135026945362148}, 'target_gap': {'Emission Peak': 24.38417103209656, 'FWHM': None}, 'objective_score': -102.57114926662203, 'information_score': 5.736698938410907, 'total_score': -102.57114926662203, 'ranking_strategy': 'best_output', 'mapped_reactants_input': '5-amino-1,10-phenanthroline|salicylic acid', 'mapped_reactants_smiles': ['Nc1cc2cccnc2c2ncccc12', 'O=C(O)c1ccccc1O'], 'nearest_known_re

In [8]:
# 3b) 지정 입력에 대한 예측 (GP posterior mean / std)
# suggest_next_experiments와 달리, 사용자가 정한 반응물·조건에 대한 예측값을 조회한다.
query_inputs = [
    {"reactants": "2,6-Diaminonaphthalene", "Temperature": 200.0},
    {"reactants": "5-amino-1,10-phenanthroline|salicylic acid", "Temperature": 250.0},
]
prediction_result = pipeline.predict_targets(artifacts, query_inputs)

for idx, row in enumerate(prediction_result.predictions, 1):
    print(f"- query_{idx}")
    print("  reactants:", row["reactants_input"])
    print("  conditions:", row["conditions"])
    print("  predicted mean:", row["predicted_target_mean"])
    print("  predicted std:", row["predicted_target_std"])
    print("  target gap:", row["target_gap"])

- query_1
  reactants: 2,6-Diaminonaphthalene
  conditions: {'Temperature': 200.0}
  predicted mean: {'Emission Peak': 514.1509776932708, 'FWHM': 96.97049329089901}
  predicted std: {'Emission Peak': 3.216219500875975, 'FWHM': 3.1060231434735095}
  target gap: {'Emission Peak': 24.15097769327076, 'FWHM': None}
- query_2
  reactants: 5-amino-1,10-phenanthroline|salicylic acid
  conditions: {'Temperature': 250.0}
  predicted mean: {'Emission Peak': 514.5155190238942, 'FWHM': 97.64560908713922}
  predicted std: {'Emission Peak': 3.228269080183387, 'FWHM': 3.134135498787738}
  target gap: {'Emission Peak': 24.51551902389417, 'FWHM': None}


In [ ]:
# 4) 신규 배치 데이터 append
# 파일이 준비되어 있지 않으면 이 셀은 건너뛰세요.
merged = dataset_service.append_csv(DATA_PATH, NEW_DATA_PATH, MERGED_OUT_PATH)
print("Merged rows:", len(merged))
print("Saved to:", MERGED_OUT_PATH)

In [ ]:
query_inputs = [
    {"reactants": "1-Aminonaphthalene", "Temperature": 300.0},
    {"reactants": "5-amino-1,10-phenanthroline|salicylic acid", "Temperature": 250.0},
]
prediction_result = pipeline.predict_targets(artifacts, query_inputs)

for idx, row in enumerate(prediction_result.predictions, 1):
    print(f"- query_{idx}")
    print("  reactants:", row["reactants_input"])
    print("  conditions:", row["conditions"])
    print("  predicted mean:", row["predicted_target_mean"])
    print("  predicted std:", row["predicted_target_std"])